# Preparação dos metadados
Carregar as outras tabelas e conferir os campos que vamos usar junto aos reviews.
Começamos por pedidos. Ao final, salva pedidos preparados e reviews com metadados em data/processed.


## 1. Carregamento


In [ ]:
from pathlib import Path
from io import BytesIO
from IPython.display import display
import pandas as pd
import kagglehub
import zipfile

DATASET = "olistbr/brazilian-ecommerce/versions/2"

def carregar_tabela(arquivo):
    caminho = Path(kagglehub.dataset_download(DATASET, path=arquivo))

    if zipfile.is_zipfile(caminho):
        with zipfile.ZipFile(caminho) as zip_file:
            conteudo = zip_file.read(arquivo)
        caminho = BytesIO(conteudo)

    return pd.read_csv(
        caminho,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8",
    )


Carregamos as oito tabelas adicionais da EDA. Geolocalização tem mais linhas e pode demorar. Nem todas entram nas junções; todas serão salvas separadamente.


In [ ]:
arquivos = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

carregar = list(arquivos)

tabelas = {}
for nome in carregar:
    print("Carregando:", nome, flush=True)
    tabelas[nome] = carregar_tabela(arquivos[nome])
    print("Linhas:", len(tabelas[nome]))

pedidos = tabelas["orders"].copy()
display(pedidos.head())


## 2. Identificadores
Precisamos de um registro por `order_id` para juntar pedidos aos reviews sem multiplicar linhas.


In [ ]:
id_vazio = pedidos["order_id"].fillna("").str.strip().eq("")
id_repetido = pedidos["order_id"].duplicated(keep=False)

print("order_id ausentes:", id_vazio.sum())
print("Linhas com order_id repetido:", id_repetido.sum())

display(pedidos.loc[id_vazio | id_repetido].head(10))


## 3. Status e datas ausentes
Uma data de entrega vazia pode fazer sentido em um pedido cancelado ou ainda não entregue. Não vamos preencher datas nem excluir pedidos automaticamente.


In [ ]:
display(pedidos["order_status"].value_counts().to_frame("pedidos"))

campos_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for campo in campos_data:
    vazio = pedidos[campo].fillna("").str.strip().eq("")
    print(f"{campo} — ausentes:", vazio.sum())


## 4. Validar o formato das datas


In [ ]:
datas_convertidas = {}

for campo in campos_data:
    datas_convertidas[campo] = pd.to_datetime(
        pedidos[campo],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )

    preenchido = pedidos[campo].fillna("").str.strip().ne("")
    invalido = preenchido & datas_convertidas[campo].isna()

    print(f"{campo} — preenchidas não convertidas:", invalido.sum())
    display(pedidos.loc[invalido, ["order_id", campo]].head(5))


### Pedidos sem data de aprovação
Conferir o status desses pedidos.


In [ ]:
# Conferir o status dos pedidos sem data de aprovação na fonte.
sem_aprovacao = (
    tabelas["orders"]["order_approved_at"]
    .fillna("")
    .str.strip()
    .eq("")
)

resumo_sem_aprovacao = (
    tabelas["orders"].loc[sem_aprovacao, "order_status"]
    .value_counts()
    .to_frame("pedidos_sem_data_aprovacao")
)

display(resumo_sem_aprovacao)


Na consulta executada, foram encontrados 160 pedidos sem data de aprovação: 141 cancelados, 5 criados e 14 entregues.

As datas permanecem ausentes, sem preenchimento artificial. Nenhum pedido é excluído por esse motivo. A data de aprovação não é usada no cálculo de atraso, que compara entrega real e previsão.


## 5. Calcular atraso
Usando entrega menos previsão, comparando apenas as datas. Valor positivo indica atraso; negativo indica antecipação.
Calculando para pedidos entregues com as duas datas preenchidas. Nos demais, o resultado fica ausente.


In [ ]:
# Não aplicar a conversão se houver datas preenchidas não reconhecidas.
for campo in campos_data:
    preenchido = pedidos[campo].fillna("").str.strip().ne("")
    if (preenchido & datas_convertidas[campo].isna()).any():
        raise ValueError(f"Revise as datas de {campo} antes de continuar.")

for campo in campos_data:
    pedidos[campo] = datas_convertidas[campo]

entrega = pedidos["order_delivered_customer_date"]
previsao = pedidos["order_estimated_delivery_date"]
compra = pedidos["order_purchase_timestamp"]

entrega_antes_compra = entrega.notna() & compra.notna() & entrega.lt(compra)
print("Entrega anterior à compra:", entrega_antes_compra.sum())
display(pedidos.loc[entrega_antes_compra, ["order_id", "order_purchase_timestamp", "order_delivered_customer_date"]].head(5))

pode_calcular = (
    pedidos["order_status"].eq("delivered")
    & entrega.notna()
    & previsao.notna()
    & ~entrega_antes_compra
)

pedidos["dias_atraso"] = (
    entrega.dt.normalize() - previsao.dt.normalize()
).dt.days.where(pode_calcular).astype("Int64")

pedidos["entrega_atrasada"] = pedidos["dias_atraso"].gt(0)

display(pedidos[["order_id", "dias_atraso", "entrega_atrasada"]].head(10))
print("Pedidos com atraso calculado:", pedidos["dias_atraso"].notna().sum())


## 6. Ler os reviews preparados


In [ ]:
raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta_saida = raiz / "data" / "processed"
pasta_docs = raiz / "docs"

reviews = pd.read_csv(
    pasta_saida / "olist_reviews_preparados.csv",
    dtype="string", keep_default_na=False, encoding="utf-8",
)
print("Reviews carregados:", len(reviews))


## 7. Preparar os campos adicionais
Usando cidade e UF do cliente, categorias e pagamentos. Itens e pagamentos podem ter várias linhas por pedido; agrupando antes da junção.
As categorias descrevem os produtos do pedido. Não indicam necessariamente qual produto foi avaliado no texto.


In [ ]:
def validar_chave(df, campo):
    if df[campo].fillna("").str.strip().eq("").any():
        raise ValueError(f"Há {campo} ausente. Revise antes da junção.")
    if df[campo].duplicated().any():
        raise ValueError(f"Há {campo} repetido. Revise antes da junção.")

def juntar_valores(serie):
    valores = serie.dropna().astype(str).str.strip()
    return " | ".join(sorted(set(valores[valores.ne("")])))

clientes = tabelas["customers"][
    ["customer_id", "customer_city", "customer_state"]
].copy()
produtos = tabelas["products"][
    ["product_id", "product_category_name"]
].copy()

validar_chave(pedidos, "order_id")
validar_chave(clientes, "customer_id")
validar_chave(produtos, "product_id")

itens = tabelas["order_items"][["order_id", "product_id"]].merge(
    produtos, on="product_id", how="left", validate="many_to_one",
)
categorias = (
    itens.groupby("order_id")["product_category_name"]
    .agg(juntar_valores)
    .rename("categorias_pedido")
    .reset_index()
)

pagamentos = tabelas["order_payments"].copy()
valores = pd.to_numeric(pagamentos["payment_value"], errors="coerce")
preenchidos = pagamentos["payment_value"].fillna("").str.strip().ne("")
if (preenchidos & valores.isna()).any():
    raise ValueError("Há valores de pagamento não reconhecidos. Revise antes de continuar.")
pagamentos["payment_value"] = valores

formas = (
    pagamentos.groupby("order_id")["payment_type"]
    .agg(juntar_valores)
    .rename("formas_pagamento")
)
# Ausência em alguma parcela impede tratar a soma como valor total conhecido.
total_pago = pagamentos.groupby("order_id")["payment_value"].agg(
    lambda s: s.sum() if s.notna().all() else float("nan")
).rename("valor_pago_pedido")
pagamento_pedido = pd.concat([formas, total_pago], axis=1).reset_index()

metadados = (
    pedidos.merge(clientes, on="customer_id", how="left", validate="many_to_one")
    .merge(categorias, on="order_id", how="left", validate="one_to_one")
    .merge(pagamento_pedido, on="order_id", how="left", validate="one_to_one")
)
assert len(metadados) == len(pedidos)
print("Pedidos com metadados:", len(metadados))
display(metadados.head())


## 8. Juntar aos reviews
O par review_id + order_id deve continuar único.


In [ ]:
for campo in ["review_id", "order_id"]:
    if reviews[campo].fillna("").str.strip().eq("").any():
        raise ValueError(f"Há {campo} ausente nos reviews.")
if reviews.duplicated(["review_id", "order_id"]).any():
    raise ValueError("Há pares review_id/order_id repetidos nos reviews.")

sobrepostos = (set(reviews.columns) & set(metadados.columns)) - {"order_id"}
if sobrepostos:
    raise ValueError(f"Revise campos sobrepostos: {sorted(sobrepostos)}")

reviews_com_metadados = reviews.merge(
    metadados, on="order_id", how="left",
    validate="many_to_one", indicator="vinculo_pedido",
)
assert len(reviews_com_metadados) == len(reviews)
assert not reviews_com_metadados.duplicated(["review_id", "order_id"]).any()

print("Reviews antes:", len(reviews))
print("Reviews depois:", len(reviews_com_metadados))
print("Sem pedido correspondente:", reviews_com_metadados["vinculo_pedido"].eq("left_only").sum())
display(reviews_com_metadados.head())


## 9. Salvar os arquivos

As oito tabelas carregadas são salvas separadamente em data/raw, sem tratamentos. Em data/processed, salvamos três CSVs: pedidos, metadados por pedido e reviews enriquecidos. O CSV de reviews preparado no outro notebook permanece intacto. 
Datas são escritas como YYYY-MM-DD HH:MM:SS.


In [ ]:
# Cópias das tabelas originais: não recebem o nome de dados tratados.
pasta_originais = raiz / "data" / "raw"
pasta_originais.mkdir(parents=True, exist_ok=True)

for nome, df in tabelas.items():
    destino = pasta_originais / arquivos[nome]
    df.to_csv(destino, index=False, encoding="utf-8")
    print("Original salvo:", destino.resolve())

pasta_saida.mkdir(parents=True, exist_ok=True)
pasta_docs.mkdir(parents=True, exist_ok=True)

saidas = {
    "olist_orders_preparados.csv": pedidos,
    "olist_metadados_pedidos.csv": metadados,
    "olist_reviews_com_metadados.csv": reviews_com_metadados,
}

for nome, df in saidas.items():
    destino = pasta_saida / nome
    df.to_csv(
        destino, index=False, encoding="utf-8",
        date_format="%Y-%m-%d %H:%M:%S",
    )
    print("Salvo:", destino.resolve())

sem_pedido = reviews_com_metadados["vinculo_pedido"].eq("left_only").sum()
resumo = f"""# Preparação dos metadados

Fonte: {DATASET}.
Pedidos preparados: {len(pedidos)}.
Reviews enriquecidos: {len(reviews_com_metadados)}.
Reviews sem pedido correspondente: {sem_pedido}.

- Datas de pedidos validadas e convertidas.
- Atraso calculado por dia para pedidos entregues com datas disponíveis.
- Entregas anteriores à compra ficam fora do cálculo de atraso.
- Atraso desconhecido permanece ausente, não recebe zero ou falso.
- Cidade e UF associadas pelo customer_id.
- Categorias reunidas por pedido, sem atribuir o comentário a um produto específico.
- Formas e valores de pagamento agrupados por pedido.
- Sem exclusão automática de pedidos cancelados ou de reviews sem correspondência.
- Junção preservou a quantidade de avaliações e o par review_id/order_id.
- As oito tabelas adicionais foram exportadas individualmente em data/raw sem tratamentos.
- Geolocalização, vendedores e tradução de categorias não foram incorporados às junções.
- Campos de texto limpo dos reviews foram preservados.
- CSV não preserva tipos; definir os tipos necessários ao carregar.

Arquivos originais em data/raw:
{chr(10).join("- " + arquivos[nome] for nome in tabelas)}

Arquivos em data/processed:
- olist_orders_preparados.csv
- olist_metadados_pedidos.csv
- olist_reviews_com_metadados.csv
"""
(pasta_docs / "resumo_metadados.md").write_text(resumo, encoding="utf-8")
print("Resumo salvo em:", pasta_docs / "resumo_metadados.md")
